# 05 - Walk-Forward Validation

The only honest way to evaluate a time-series model.

**Problem with a single train/test split**: you get one number, on one
slice of history. Maybe that slice was kind to your model by luck.

**Walk-forward** answers: does the model work **repeatedly**, across
different market regimes?

Method:

1. Train on days `[t - N : t]`
2. Test on days `[t : t + M]`
3. Slide `t` forward by `M`
4. Repeat until end of data

Each window is fully out-of-sample relative to its own training set.

This notebook:

- Runs walk-forward for 3 models: baseline, Ridge, LightGBM
- Plots RMSE and direction accuracy per window
- Shows distribution across windows (mean, std, worst)
- Saves `reports/predictions/walkforward.json`


## Setup

In [6]:
# --- setup --------------------------------------------------------
import sys, json
sys.path.insert(0, r"D:\Projects\done\eth_bot\eth_bot_v2")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

from analysis.loader import load_ohlcv
from analysis.indicators import add_all

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (12, 4)

SEED = 42
np.random.seed(SEED)

PRED_DIR = Path(r"D:\Projects\done\eth_bot\eth_bot_v2") / "reports" / "predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

try:
    import lightgbm as lgb
    HAS_LGB = True
    print("lightgbm: installed")
except ImportError:
    HAS_LGB = False
    print("lightgbm: not installed (pip install lightgbm)")

lightgbm: installed


## 1. Features

Same 9 features used everywhere else, plus next-day return target.

In [7]:
def make_features(horizon: int = 1) -> pd.DataFrame:
    df = load_ohlcv()
    df = add_all(df).copy()

    df["ret_1"]  = df["close"].pct_change(1)
    df["ret_5"]  = df["close"].pct_change(5)
    df["ret_20"] = df["close"].pct_change(20)
    df["vol_20"] = df["ret_1"].rolling(20).std()

    df["target"] = df["close"].pct_change(horizon).shift(-horizon)
    return df.dropna()


FEATURES = [
    "ret_1", "ret_5", "ret_20", "vol_20",
    "rsi_14", "macd_hist", "atr_14",
    "sma_50", "sma_200",
]

df = make_features()
df = df[FEATURES + ["target"]].dropna()
print(f"rows: {len(df)}   features: {len(FEATURES)}")
print(f"range: {df.index.min().date()} -> {df.index.max().date()}")

rows: 1989   features: 9
range: 2021-04-22 -> 2026-10-01


## 2. Walk-forward function

Parameters:

- `train_days = 500` — roughly 1.4 years of history per fold
- `test_days = 60` — 2 months forward
- step = `test_days` (60), so windows are non-overlapping

With 2188 rows, that gives about **28 windows** — plenty for statistics.

In [8]:
def walk_forward(df, model_factory, train_days=500, test_days=60):
    rows = []
    start = train_days
    while start + test_days <= len(df):
        train = df.iloc[start - train_days : start]
        test  = df.iloc[start : start + test_days]

        X_tr, y_tr = train[FEATURES].values, train["target"].values
        X_te, y_te = test[FEATURES].values,  test["target"].values

        model = model_factory()
        model.fit(X_tr, y_tr)
        p = model.predict(X_te)

        rows.append({
            "start":  test.index[0].date(),
            "end":    test.index[-1].date(),
            "rmse":   float(np.sqrt(mean_squared_error(y_te, p))),
            "mae":    float(np.abs(p - y_te).mean()),
            "dir":    float(((p > 0) == (y_te > 0)).mean()),
            "n_test": int(len(y_te)),
        })
        start += test_days
    return pd.DataFrame(rows)


# baseline in the same format: predict 0 for every test day
def baseline_factory():
    class Zero:
        def fit(self, X, y): return self
        def predict(self, X): return np.zeros(len(X))
    return Zero()


print("walk-forward function ready")

walk-forward function ready


## 3. Run all three models

Runs each model across every walk-forward window.

Takes ~10-20 seconds total.

In [9]:
results = {}

results["baseline"] = walk_forward(df, baseline_factory)
print(f"baseline : {len(results['baseline'])} windows")

results["ridge"] = walk_forward(df, lambda: Ridge(alpha=1.0))
print(f"ridge    : {len(results['ridge'])} windows")

if HAS_LGB:
    results["lgbm"] = walk_forward(
        df,
        lambda: lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05,
                                  max_depth=4, num_leaves=31,
                                  random_state=SEED, verbose=-1)
    )
    print(f"lgbm     : {len(results['lgbm'])} windows")

baseline : 24 windows


ridge    : 24 windows


OSError: exception: access violation reading 0x0000000000000000

## 4. Aggregate results

For each model, summarise across all windows:

- `rmse_mean` — average RMSE
- `rmse_std` — variation across windows (lower is more stable)
- `rmse_worst` — worst window (how bad can it get)
- `dir_mean` — average direction accuracy
- `win_rate` — fraction of windows that beat 50% direction

In [ ]:
def summarize(name, res):
    return {
        "model":      name,
        "windows":    len(res),
        "rmse_mean":  float(res.rmse.mean()),
        "rmse_std":   float(res.rmse.std(ddof=0)),
        "rmse_best":  float(res.rmse.min()),
        "rmse_worst": float(res.rmse.max()),
        "mae_mean":   float(res.mae.mean()),
        "dir_mean":   float(res.dir.mean()),
        "dir_std":    float(res.dir.std(ddof=0)),
        "win_rate":   float((res.dir > 0.5).mean()),
    }


summary = pd.DataFrame([summarize(name, res) for name, res in results.items()])
summary

## 5. Compare to a single-split baseline

Take the last `test_days = 60` window from each model — this is the
equivalent of the single train/test split used in earlier notebooks.

Then compare with the walk-forward mean.

In [ ]:
last_vals = []
for name, res in results.items():
    last = res.iloc[-1]
    last_vals.append({
        "model":        name,
        "last_rmse":    last.rmse,
        "wf_mean_rmse": res.rmse.mean(),
        "diff":         last.rmse - res.rmse.mean(),
    })

pd.DataFrame(last_vals)

**Reading the table**: if `diff` is large and positive, the last window was
unusually hard (or easy) for that model. That's exactly the risk of using
a single split — you might have reported a number that doesn't
represent the model's real behaviour.

## 6. Plot RMSE per window

Each point is one walk-forward window.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for name, res in results.items():
    ax.plot(res.start, res.rmse, marker="o", ms=3, lw=1.0, label=name)
ax.set_title("RMSE per walk-forward window")
ax.set_ylabel("RMSE")
ax.set_xlabel("window start")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7. Plot direction accuracy per window

The 50% line is a coin flip.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
for name, res in results.items():
    ax.plot(res.start, res.dir, marker="o", ms=3, lw=1.0, label=name)
ax.axhline(0.5, ls="--", color="gray", lw=0.8)
ax.set_title("direction accuracy per walk-forward window")
ax.set_ylabel("accuracy")
ax.set_xlabel("window start")
ax.set_ylim(0.3, 0.7)
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 8. Distribution of RMSE across windows

Boxplot shows spread. Tight boxes = consistent model.
Wide boxes = model performance depends on which market regime it meets.

In [ ]:
import pandas as pd

all_rmse = pd.concat(
    [res.assign(model=name)[["model", "rmse", "dir"]] for name, res in results.items()],
    ignore_index=True,
)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))

all_rmse.boxplot(column="rmse", by="model", ax=a1)
a1.set_title("RMSE distribution")
a1.set_xlabel("")

all_rmse.boxplot(column="dir", by="model", ax=a2)
a2.axhline(0.5, ls="--", color="gray")
a2.set_title("direction accuracy distribution")
a2.set_xlabel("")

plt.suptitle("")
plt.tight_layout()
plt.show()

## 9. Head-to-head: how often does each model beat the baseline?

For each window, did the model's RMSE come in below the baseline's
on the same window? This is the **honest measure of edge**.

In [ ]:
base = results["baseline"].set_index("start")["rmse"]
head = []
for name, res in results.items():
    if name == "baseline":
        continue
    r = res.set_index("start")["rmse"]
    common = base.index.intersection(r.index)
    wins = (r.loc[common] < base.loc[common]).sum()
    total = len(common)
    head.append({
        "model":       name,
        "wins":        int(wins),
        "losses":      int(total - wins),
        "beat_rate":   float(wins / total),
    })
pd.DataFrame(head)

**This is the number that matters.**

If a model beats the baseline in 60%+ of windows, it has a real edge.
If it wins 45% of the time, it's noise dressed as a model.

## 10. Save results

In [ ]:
out = {
    "train_days": 500,
    "test_days":  60,
    "features":   FEATURES,
    "models":     {name: {
        "rmse_mean":  float(res.rmse.mean()),
        "rmse_std":   float(res.rmse.std(ddof=0)),
        "dir_mean":   float(res.dir.mean()),
        "win_rate":   float((res.dir > 0.5).mean()),
        "windows":    int(len(res)),
    } for name, res in results.items()},
}

path = PRED_DIR / "walkforward.json"
path.write_text(json.dumps(out, indent=2), encoding="utf-8")
print("wrote", path)
print()
print(json.dumps(out["models"], indent=2))

## Interpreting the results

| What you see | What it means |
|---|---|
| Model beats baseline > 55% of windows | real edge — keep it |
| Model beats baseline 45-55% of windows | coin flip — no edge |
| Model loses to baseline most windows | harmful — don't use it |
| Ridge and LGBM behave nearly identically | signal is basically linear |
| RMSE std is huge | model performance depends on regime |

**Expected outcome**: for daily crypto with these features, no model
beats the baseline reliably. That's normal. The important result is
**knowing** it, rather than being fooled by a lucky single-split number.

## Next steps

- **06_feature_importance.ipynb** — SHAP for the tree model: which features actually drive predictions?
- **07_model_comparison.ipynb** — combine all results into a leaderboard
- **Adding features** — merge `funding_rate`, `open_interest`, `fear_greed` and re-run. This is more likely to help than any model change.
